# Model training

Trains CatBoost on `train_clean.csv` / `test_clean.csv` (produced by the EDA notebook's cleaning + feature engineering steps) and writes `predictions.csv`. Equivalent to `import pandas as pd.py`.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor

## Feature configuration

In [2]:
ID_COLUMNS = ['Item_Identifier', 'Outlet_Identifier']
NUMERIC_COLUMNS = ['Item_Weight', 'Item_Visibility', 'Item_MRP', 'Outlet_Establishment_Year', 'Outlet_Age', 'Visibility_Ratio']
CATEGORICAL_COLUMNS = ['Item_Fat_Content', 'Item_Type', 'Item_Category', 'MRP_Band', 'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type']
FEATURES = NUMERIC_COLUMNS + CATEGORICAL_COLUMNS

## Load the cleaned data

Cleaning and feature engineering (fat content, weight/visibility/size imputation, `Item_Category`, `Visibility_Ratio`) are done in the EDA notebook, which writes these files.

In [3]:
test = pd.read_csv('test_clean.csv')
train = pd.read_csv('train_clean.csv')
print('test:', test.shape, '-> missing:', test.isnull().sum().sum())
print('train:', train.shape, '-> missing:', train.isnull().sum().sum())

test: (5681, 15) -> missing: 0
train: (8523, 16) -> missing: 0


In [4]:
# Keep the ID columns for the submission file before they get dropped from the feature set.
test_ids = test[ID_COLUMNS].reset_index(drop=True)

## Validate on a held-out split

In [5]:
# Held-out split so RMSE reflects unseen data, not just training fit.
X_train, X_val, y_train, y_val = train_test_split(
    train[FEATURES], train['Item_Outlet_Sales'], test_size=0.2, random_state=25,
)

validator = CatBoostRegressor(cat_features=CATEGORICAL_COLUMNS, verbose=False, random_state=25, depth=6, learning_rate=0.03, iterations=300)
validator.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)
val_rmse = root_mean_squared_error(y_val, np.clip(validator.predict(X_val), 0, None))
print('validation RMSE:', val_rmse)

validation RMSE: 1100.2947665947547


## Fit the final model and predict on test

In [6]:
# Refit on the full training set for the final model used to predict on test.
model = CatBoostRegressor(cat_features=CATEGORICAL_COLUMNS, verbose=False, random_state=25, depth=6, learning_rate=0.03, iterations=300)
model.fit(train[FEATURES], train['Item_Outlet_Sales'])

train_pred = np.clip(model.predict(train[FEATURES]), 0, None)
train_rmse = root_mean_squared_error(train['Item_Outlet_Sales'], train_pred)
print('train RMSE:', train_rmse)

train RMSE: 1059.7241102875093


In [7]:
# Sales can't be negative, so clip away any stray negative predictions from the linear trend.
test['Item_Outlet_Sales'] = np.clip(model.predict(test[FEATURES]), 0, None)
print(test[['Item_Outlet_Sales']].describe())

       Item_Outlet_Sales
count        5681.000000
mean         2176.838668
std          1320.834558
min            41.710979
25%          1230.347439
50%          1955.273203
75%          3024.371193
max          6332.572017


## Save predictions

In [8]:
predictions = pd.concat([test_ids, test['Item_Outlet_Sales']], axis=1)
predictions.to_csv('predictions.csv', index=False)